In [ ]:
# 이미지 증강 사용

```
https://www.tensorflow.org/tutorials/images/data_augmentation?hl=ko
https://hwiyong.tistory.com/402
```

In [ ]:
# 전처리 레이어
from keras import layers

resize_and_rescale = tf.keras.Sequential([
  layers.Resizing(224, 224),
  layers.Rescaling(1./255),
], name='resize_rescaling')

In [ ]:
# 데이터 증강 레이어를 구성한다.

data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.2),
        # layers.RandomZoom(0.2),
        # layers.RandomContrast(0.2),
        # layers.RandomBrightness(0.2),
        # layers.RandomTranslation(0.2, 0.2),
        # Add more data augmentation layers as needed

    ], name='augmentation'
)

In [ ]:
# 데이터 증강 시각화

plt.figure(figsize=(10, 10))
for images, _ in train_ds.take(1):
    for i in range(9):
        augmented_images = data_augmentation(images)
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(augmented_images[0].numpy().astype("uint8"))
        plt.axis("off")

-  모델에 데이터 증강을 사용하는 방법  
이 방법을 사용하면 테스트 데이터에도 그냥 별도의 처리없이 바로 예측에 적용할 수 있다.

```
inputs = keras.Input(shape=input_shape)
x = data_augmentation(inputs)
x = layers.experimental.preprocessing.Rescaling(1./255)(x)
...  # Rest of the model

```

In [ ]:
# generate dummy data
input_dim = (224,224,3)

In [ ]:
from keras import layers

def build_model(input_shape):
    # 함수형API 모델을 작성할 때는 Input이 여기에 있어야 하네...
    inputs = layers.Input(shape=input_shape)

    # 본격적으로 모델에 입력하기 전에 여기서 augmentation이 진행됩니다.
    # inference time에는 동작하지 않습니다.
    x = data_augmentation(inputs)
    x = resize_and_rescale(x)
    # [0, 1] 변환을 위해 Rescaling Layer를 활용합니다.
    x = layers.Rescaling(1.0 / 255)(x)

    # CNN 모델
    x = layers.Conv2D(32, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, activation='relu')(x)
    x = layers.MaxPooling2D()(x)

    #  x = layers.GlobalAveragePooling2D()(x) 를 사용할 수도 있슴
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)
    model = keras.Model(inputs, outputs)
    return model

In [ ]:
# 이 모델은 서머리를 보기위해서는 빌드를 해야한다.
model = build_model(input_dim)
model.summary()

## 모델 학습

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy",
              metrics=["accuracy"])


In [ ]:
EPOCHS = 3

history = model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=EPOCHS
)

## 예측 및 평가

```
참고
https://hwiyong.tistory.com/402
```

In [ ]:
# inference time에는 Dropout과 data augmentation이 비활성화 상태가 됩니다.
# 하지만 Rescaling layer는 그대로 활용되기 때문에 테스트용 데이터로 추론을
# 진행할 때 올바른 결과를 얻을 수 있습니다.

In [ ]:
results = model.evaluate(test_ds)
results

In [ ]:
# 한개의 이미지를 예측한다

img = keras.preprocessing.image.load_img(
    '/content/cats_and_dogs/test/cats/cat.2300.jpg',
    target_size=(img_height, img_width)
)

img

In [ ]:
img_array = keras.preprocessing.image.img_to_array(img)

# 축추가 - 하나의 이미지를 예측하더라도 배치형태이어야 한다
# img_array = tf.expand_dims(img_array, 0)
img_array = img_array.reshape(-1, 224, 224, 3)  # Create batch axis
img_array.shape


In [ ]:
y_pred = model.predict(img_array)
score = y_pred[0]
score

# 0.5보다 작으면 고양이, 크면 강아지

# END